# Notebook 04 — Chatbot Completo Integrado

**Curso:** Minería de Textos | **Proyecto 3** | CUC

---

## Orden de ejecución
```
Notebook 02 → Fine-Tuning + corpus etiquetado
Notebook 03 → RAG Pipeline + índice FAISS
Notebook 04 → Chatbot integrado (este notebook)
```

## Mejoras implementadas
- Filtro de emoción solo activo si score >= 70% (evita filtros incorrectos)
- Generador Flan-T5 en dos pasos: genera en inglés → traduce al español
- Memoria conversacional de los últimos 5 turnos


In [1]:
import sys, json
sys.path.append('../app')
sys.path.append('../src')

from pathlib import Path

from src.mongo_utils      import mongo_utils
from src.finetuning_utils import finetuning_utils
from src.rag_utils        import rag_utils
from src.chatbot_engine   import chatbot_engine

print('Librerías cargadas.')

C:\Users\pmari\OneDrive\Para Revisar\Documentos\Pablo\Cuc\2026\Mineria de Textos\chatbot-musical-inteligente\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Librerías cargadas.


## 1. Inicializar sistema completo

In [2]:
db_conexion = mongo_utils()
finetuning  = finetuning_utils()
rag         = rag_utils()

if db_conexion.verificar_conexion():
    canciones_raw     = db_conexion.cargar_canciones()
    corpus_etiquetado = finetuning.etiquetar_corpus_con_modelo(canciones_raw)
    rag.inicializar(corpus_etiquetado)
    bot = chatbot_engine()
    print('\nSistema listo: Fine-Tuning + RAG + Chatbot integrados.')
else:
    print('ERROR: No se pudo conectar a MongoDB Atlas.')

2026-04-23 01:34:12 | INFO     | mongo_utils | Conectando a MongoDB Atlas | DB: analisisMusical | Col: analisisMusical
2026-04-23 01:34:14 | INFO     | mongo_utils | Conexión verificada correctamente.
2026-04-23 01:34:14 | INFO     | mongo_utils | Cargando canciones | limite=None | solo_con_letra=True
2026-04-23 01:34:15 | INFO     | mongo_utils | Canciones cargadas: 6940
2026-04-23 01:34:15 | INFO     | EmotionClassifier | Cargando corpus etiquetado desde caché...
2026-04-23 01:34:15 | INFO     | EmotionClassifier | 6526 canciones cargadas con emoción.
2026-04-23 01:34:15 | INFO     | rag_utils | Caché completo encontrado. Cargando RAG desde disco...
2026-04-23 01:34:15 | INFO     | rag_utils | Cargando embeddings desde caché...
2026-04-23 01:34:15 | INFO     | rag_utils | Embeddings cargados: (70264, 384) | Chunks: 70264
2026-04-23 01:34:15 | INFO     | rag_utils | Cargando índice FAISS desde caché...
2026-04-23 01:34:15 | INFO     | rag_utils | Índice cargado: 70264 vectores, dim 38

Loading weights: 100%|██████████| 558/558 [00:00<00:00, 1755.85it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


2026-04-23 01:34:25 | INFO     | chatbot_engine | Flan-T5 cargado correctamente.

Sistema listo: Fine-Tuning + RAG + Chatbot integrados.


## 2. Prueba interactiva con emoción y umbral de confianza

El filtro de emoción solo se activa si el clasificador tiene >= 70% de confianza.
Esto evita filtros incorrectos como clasificar 'alegría' como 'rabia' con baja confianza.


In [3]:
def chat(pregunta):
    resultado = bot.responder(pregunta)
    emocion   = resultado.get('emocion')
    print(f'Usuario: {pregunta}')
    if emocion:
        activo = emocion['score'] >= 0.70
        print(f'[Emoción: {emocion["emocion"]} ({emocion["score"]:.0%}) '
              f'| Filtro: {"activo" if activo else "inactivo (score bajo)"}]')
    print(f'Bot: {resultado["respuesta"]}')
    fuentes = [c['titulo'] for c in resultado['chunks'][:3]]
    print(f'[Fuentes: {", ".join(fuentes)}]')
    print()
    return resultado

# Conversación de prueba con memoria
chat('Estoy triste, que cancion me recomiendas?')
chat('Dame otra similar.')
chat('De que artista es la primera que mencionaste?')

2026-04-23 01:34:26 | INFO     | chatbot_engine | Pregunta recibida: 'Estoy triste, que cancion me recomiendas?'


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 3385.52it/s]


2026-04-23 01:34:28 | INFO     | EmotionClassifier | Clasificador cargado para inferencia.
2026-04-23 01:34:29 | INFO     | rag_utils | Cargando modelo de embeddings: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2595.16it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


2026-04-23 01:34:36 | INFO     | rag_utils | Modelo cargado. Dimensión: 384


C:\Users\pmari\OneDrive\Para Revisar\Documentos\GitHub\chatbot-musical-inteligente\src\rag_utils.py:172: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  dim = self._modelo_emb.get_sentence_embedding_dimension()


2026-04-23 01:34:39 | INFO     | chatbot_engine | Respuesta generada (119 chars)
Usuario: Estoy triste, que cancion me recomiendas?
[Emoción: tristeza (27%) | Filtro: inactivo (score bajo)]
Bot: Te recomiendo "Farewell" de Rihanna, una canción de pop con emoción de rabia. También puedes escuchar "Stan" de Eminem.
[Fuentes: Farewell, Stan, So Sick]

2026-04-23 01:34:39 | INFO     | chatbot_engine | Pregunta recibida: 'Dame otra similar.'
2026-04-23 01:34:41 | INFO     | chatbot_engine | Respuesta generada (158 chars)
Usuario: Dame otra similar.
[Emoción: rabia (24%) | Filtro: inactivo (score bajo)]
Bot: Te recomiendo "One Time (J-Stax Remix)" de Justin Bieber, una canción de pop con emoción de nostalgia. También puedes escuchar "One Right Now" de Post Malone.
[Fuentes: One Time (J-Stax Remix), One Right Now, MIC Drop]

2026-04-23 01:34:41 | INFO     | chatbot_engine | Pregunta recibida: 'De que artista es la primera que mencionaste?'
2026-04-23 01:34:42 | INFO     | chatbot_engine | Res

{'respuesta': 'Te recomiendo "Feminism (Quotes)" de Ariana Grande, una canción de pop con emoción de nostalgia. También puedes escuchar "Make Things Right" de Drake.',
 'chunks': [{'texto': "think a male artist would be in this position right now sorry if i'm speaking about something that i'm passionate about i'm willing to take the brunt for fighting in what i believe in and my fellow women are definitely something that i will always be one of the first",
   'titulo': 'Feminism (Quotes)',
   'artista': 'Ariana Grande',
   'genero': 'pop',
   'anio': 2017,
   'idioma': 'en',
   'emocion': 'nostalgia',
   'emocion_score': 0.942,
   'score': 0.14652913808822632},
  {'texto': "drake now up north there's five artists deservin' a listen and i'm one of em the other four you know who you are but if you gotta think through chances are that it ain't you i singlehandedly carry out what you can't do and see i take a couple of",
   'titulo': 'Make Things Right',
   'artista': 'Drake',
   'genero':

## 3. Generar 10 conversaciones de prueba

In [4]:
bot.limpiar_historial()

conversaciones = [
    # Factuales
    'Que canciones hablan de desamor?',
    'Que artistas de pop hay en el corpus?',
    # Emocionales
    'Estoy muy feliz hoy, que cancion me recomiendas?',
    'Necesito algo para llorar un rato.',
    # Comparativas
    'Que diferencia hay entre una balada triste y una alegre?',
    # Seguimiento (prueba de memoria)
    'Dame otra cancion parecida a la ultima que mencionaste.',
    'Y esa de quien es?',
    # Por género
    'Dame una cancion de pop con emocion de amor.',
    # Fuera de dominio (el bot debe decir que no sabe)
    'Cuanto cuesta un vuelo a Madrid?',
    'Quien gano el mundial de futbol en 2022?',
]

registro = []
for i, pregunta in enumerate(conversaciones, 1):
    print(f'--- Conversación {i} ---')
    resultado = chat(pregunta)
    registro.append({
        'id':       i,
        'pregunta': pregunta,
        'respuesta': resultado['respuesta'],
        'emocion_detectada': resultado.get('emocion'),
        'fuentes': [
            {'titulo': c['titulo'], 'artista': c['artista'],
             'emocion': c['emocion'], 'score': c['score']}
            for c in resultado['chunks']
        ],
    })

# Guardar en metricas.json
Path('../resultados').mkdir(exist_ok=True)
metricas_path = Path('../resultados/metricas.json')
metricas_existentes = {}
if metricas_path.exists():
    with open(metricas_path, 'r', encoding='utf-8') as f:
        metricas_existentes = json.load(f)

metricas_existentes['conversaciones_prueba'] = registro
with open(metricas_path, 'w', encoding='utf-8') as f:
    json.dump(metricas_existentes, f, ensure_ascii=False, indent=2)
print('\n10 conversaciones guardadas en resultados/metricas.json')

2026-04-23 01:34:42 | INFO     | chatbot_engine | Historial conversacional limpiado.
--- Conversación 1 ---
2026-04-23 01:34:42 | INFO     | chatbot_engine | Pregunta recibida: 'Que canciones hablan de desamor?'
2026-04-23 01:34:44 | INFO     | chatbot_engine | Respuesta generada (143 chars)
Usuario: Que canciones hablan de desamor?
[Emoción: tristeza (39%) | Filtro: inactivo (score bajo)]
Bot: Te recomiendo "So Sick" de Justin Bieber, una canción de pop con emoción de tristeza. También puedes escuchar "I Wanna Be With U" de Lady Gaga.
[Fuentes: So Sick, I Wanna Be With U, Stan (Live from the 43rd Annual Grammy Awards)]

--- Conversación 2 ---
2026-04-23 01:34:44 | INFO     | chatbot_engine | Pregunta recibida: 'Que artistas de pop hay en el corpus?'
2026-04-23 01:34:45 | INFO     | chatbot_engine | Respuesta generada (149 chars)
Usuario: Que artistas de pop hay en el corpus?
[Emoción: rabia (23%) | Filtro: inactivo (score bajo)]
Bot: Te recomiendo "The Power of Pop" de Taylor Swift, u

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


2026-04-23 01:34:56 | INFO     | chatbot_engine | Respuesta generada (146 chars)
Usuario: Dame una cancion de pop con emocion de amor.
[Emoción: amor (46%) | Filtro: inactivo (score bajo)]
Bot: Te recomiendo "Trivia 起: Just Dance" de BTS (방탄소년단), una canción de k pop con emoción de alegria. También puedes escuchar "Fall" de Justin Bieber.
[Fuentes: Trivia 起: Just Dance, Fall, I Wanna Be With U]

--- Conversación 9 ---
2026-04-23 01:34:56 | INFO     | chatbot_engine | Pregunta recibida: 'Cuanto cuesta un vuelo a Madrid?'
2026-04-23 01:34:58 | INFO     | chatbot_engine | Respuesta generada (20 chars)
Usuario: Cuanto cuesta un vuelo a Madrid?
[Emoción: tristeza (28%) | Filtro: inactivo (score bajo)]
Bot: This is about music.
[Fuentes: ​​rockstar (Remix), Airplane pt.2, Te Dejo Madrid]

--- Conversación 10 ---
2026-04-23 01:34:58 | INFO     | chatbot_engine | Pregunta recibida: 'Quien gano el mundial de futbol en 2022?'
2026-04-23 01:35:04 | INFO     | chatbot_engine | Respuesta generada (

## 4. Prueba de memoria conversacional

In [5]:
bot.limpiar_historial()
print('=== PRUEBA DE MEMORIA CONVERSACIONAL ===')
chat('Dame una cancion triste de pop.')
chat('Puedes decirme mas sobre esa cancion?')
chat('Del mismo artista hay algo mas alegre?')
print(f'Turnos en memoria: {len(bot.historial)}')

2026-04-23 01:35:04 | INFO     | chatbot_engine | Historial conversacional limpiado.
=== PRUEBA DE MEMORIA CONVERSACIONAL ===
2026-04-23 01:35:04 | INFO     | chatbot_engine | Pregunta recibida: 'Dame una cancion triste de pop.'
2026-04-23 01:35:06 | INFO     | chatbot_engine | Respuesta generada (139 chars)
Usuario: Dame una cancion triste de pop.
[Emoción: amor (36%) | Filtro: inactivo (score bajo)]
Bot: Te recomiendo "Bad Guy" de Eminem, una canción de hip hop con emoción de nostalgia. También puedes escuchar "Sad Serenade" de Selena Gomez.
[Fuentes: Bad Guy, Sad Serenade, Die Alone]

2026-04-23 01:35:06 | INFO     | chatbot_engine | Pregunta recibida: 'Puedes decirme mas sobre esa cancion?'
2026-04-23 01:35:09 | INFO     | chatbot_engine | Respuesta generada (166 chars)
Usuario: Puedes decirme mas sobre esa cancion?
[Emoción: amor (23%) | Filtro: inactivo (score bajo)]
Bot: Te recomiendo "SÚBEME LA RADIO" de Enrique Iglesias, una canción de pop latino con emoción de tristeza. Tambi

## 5. Análisis del umbral de confianza

Verificar cuántas preguntas activan el filtro de emoción (score >= 70%).


In [11]:
preguntas_test = [
    'Estoy muy triste hoy, me rompieron el corazón y no puedo dejar de llorar',
    'Quiero bailar y celebrar, estoy muy feliz y con mucha energía',
    'Me siento nostálgico recordando el pasado y los viejos tiempos',
    'Dame algo energético y alegre para una fiesta con amigos',
    'Siento mucha rabia y odio, me traicionaron y quiero desahogarme',
    'Busco una canción romántica de amor para dedicarle a alguien especial',
]

print('ANÁLISIS DE UMBRAL DE CONFIANZA (>= 70% activa filtro)')
print('=' * 60)
for p in preguntas_test:
    res = finetuning.predecir_emocion(p)
    if res:
        activo = '✅ ACTIVO' if res['score'] >= 0.70 else '⚠️  inactivo'
        print(f'{activo} | {res["emocion"]:12s} {res["score"]:.0%} | "{p[:50]}..."')

ANÁLISIS DE UMBRAL DE CONFIANZA (>= 70% activa filtro)
✅ ACTIVO | tristeza     94% | "Estoy muy triste hoy, me rompieron el corazón y no..."
✅ ACTIVO | alegria      89% | "Quiero bailar y celebrar, estoy muy feliz y con mu..."
⚠️  inactivo | nostalgia    56% | "Me siento nostálgico recordando el pasado y los vi..."
✅ ACTIVO | alegria      77% | "Dame algo energético y alegre para una fiesta con ..."
⚠️  inactivo | rabia        51% | "Siento mucha rabia y odio, me traicionaron y quier..."
✅ ACTIVO | amor         77% | "Busco una canción romántica de amor para dedicarle..."


In [7]:
# Descomenta para lanzar desde el notebook:
# import subprocess
# subprocess.Popen(['python', '../app/chatbot_app.py'])
# print('App lanzada en http://localhost:8050')